# Object Detection for Plant Cropping

This notebook explores plant detection and cropping using YOLO-based models.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import sys

# Add src to path
sys.path.append('../src')

from detection import PlantDetector, SimplePlantDetector
from preprocessing import PlantCropper

plt.rcParams['figure.figsize'] = (15, 10)

## Simple Plant Detection (Color-based)

In [ ]:
# Initialize simple detector
simple_detector = SimplePlantDetector(method='green_threshold')

# Load a sample image
sample_image_path = '../data/raw/full_plant_images/sample.jpg'  # Replace with actual path

# For demonstration, create a synthetic plant image
def create_synthetic_plant_image():
    """Create a synthetic plant-like image for testing"""
    img = np.ones((400, 400, 3), dtype=np.uint8) * 200  # Light background
    
    # Add green plant-like region
    center = (200, 200)
    cv2.circle(img, center, 150, (34, 139, 34), -1)  # Dark green
    cv2.circle(img, (150, 150), 80, (50, 205, 50), -1)  # Light green
    cv2.circle(img, (250, 250), 60, (0, 100, 0), -1)  # Very dark green
    
    # Add some noise
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    img = cv2.add(img, noise)
    
    return img

test_image = create_synthetic_plant_image()

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
plt.title('Synthetic Plant Image')
plt.axis('off')
plt.show()

In [ ]:
# Detect plant region
mask = simple_detector.detect(test_image)

# Get bounding box
x1, y1, x2, y2 = simple_detector.get_bounding_box(mask)
print(f"Bounding box: ({x1}, {y1}, {x2}, {y2})")

# Visualize mask
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Plant Mask')
axes[1].axis('off')

# Draw bounding box on original
img_with_box = test_image.copy()
cv2.rectangle(img_with_box, (x1, y1), (x2, y2), (0, 255, 0), 3)
axes[2].imshow(cv2.cvtColor(img_with_box, cv2.COLOR_BGR2RGB))
axes[2].set_title('With Bounding Box')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Plant Cropping

In [ ]:
# Initialize cropper
cropper = PlantCropper(method='color_threshold')

# Crop plant
cropped_image, bbox = cropper.crop_plant(test_image)

print(f"Original size: {test_image.shape}")
print(f"Cropped size: {cropped_image.shape}")
print(f"Bounding box: {bbox}")

# Visualize cropping
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(cropped_image, cv2.COLOR_BGR2RGB))
axes[1].set_title('Cropped Plant')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## YOLO-based Plant Detection (if model available)

In [ ]:
# Initialize YOLO detector (requires trained model)
# Uncomment and modify path if you have a trained model

# yolo_detector = PlantDetector(
#     model_path='../experiments/checkpoints/plant_detection/best.pt',
#     confidence_threshold=0.5,
#     device='cuda' if torch.cuda.is_available() else 'cpu'
# )

# detections = yolo_detector.detect(test_image, return_crops=True)
# print(f"Number of detections: {len(detections)}")

# for i, det in enumerate(detections):
#     print(f"Detection {i+1}:")
#     print(f"  Bounding box: {det['bbox']}")
#     print(f"  Confidence: {det['confidence']:.2f}")
#     print(f"  Class ID: {det['class_id']}")

## Batch Processing for Dataset

In [ ]:
# Process multiple images (demonstration)
def process_batch_images(image_list, cropper):
    """Process a batch of images for cropping"""
    results = []
    
    for img in image_list:
        cropped, bbox = cropper.crop_plant(img)
        results.append({
            'original_shape': img.shape,
            'cropped_shape': cropped.shape,
            'bbox': bbox
        })
    
    return results

# Create multiple synthetic images for testing
test_images = [create_synthetic_plant_image() for _ in range(3)]

# Process batch
batch_results = process_batch_images(test_images, cropper)

for i, result in enumerate(batch_results):
    print(f"Image {i+1}:")
    print(f"  Original: {result['original_shape']}")
    print(f"  Cropped: {result['cropped_shape']}")
    print(f"  BBox: {result['bbox']}")
    print()

## Detection Quality Metrics

In [ ]:
# Calculate detection quality metrics
def calculate_detection_quality(mask):
    """Calculate quality metrics for detection mask"""
    total_pixels = mask.shape[0] * mask.shape[1]
    plant_pixels = np.sum(mask > 0)
    
    coverage_ratio = plant_pixels / total_pixels
    
    # Calculate mask solidity (area / convex hull area)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        hull = cv2.convexHull(largest_contour)
        hull_area = cv2.contourArea(hull)
        solidity = area / hull_area if hull_area > 0 else 0
    else:
        solidity = 0
    
    return {
        'coverage_ratio': coverage_ratio,
        'solidity': solidity,
        'plant_pixels': plant_pixels,
        'total_pixels': total_pixels
    }

# Calculate metrics for our test image
quality_metrics = calculate_detection_quality(mask)
print("Detection Quality Metrics:")
for key, value in quality_metrics.items():
    print(f"  {key}: {value:.4f}")

## Summary

In [ ]:
print("Object Detection Summary:")
print("- Simple color-based detection works for plants with green foliage")
print("- YOLO-based detection requires trained model for better accuracy")
print("- Plant cropping reduces background noise for classification")
print("- Detection quality metrics help evaluate cropping performance")